In [9]:
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
from scipy.fftpack import dct
import torchvision.transforms as transforms

In [10]:
class CIFakeTestDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root = root
        self.transform = transform
        self.image_paths = sorted([
            os.path.join(root, f) for f in os.listdir(root)
            if f.endswith(".png")
        ])

    def __len__(self):
        return len(self.image_paths)

    def _compute_dct_features(self, image, block_size=16):
        gray = np.array(image.convert("L")) / 255.0
        h, w = gray.shape
        dct_full = dct(dct(gray.T, norm='ortho').T, norm='ortho')
        dct_block = dct_full[:block_size, :block_size].flatten()
        dct_vec = dct_block[:256] if len(dct_block) >= 256 else np.pad(dct_block, (0, 256 - len(dct_block)))
        return torch.tensor(dct_vec, dtype=torch.float32)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        index = int(os.path.splitext(os.path.basename(img_path))[0]) 

        try:
            image = Image.open(img_path).convert("RGB")
            dct_features = self._compute_dct_features(image)
        except Exception:
            image = Image.new("RGB", (256, 256), color=(255, 0, 0))
            dct_features = torch.zeros(256)

        if self.transform:
            image = self.transform(image)

        return {
            "index": index,
            "image": image,
            "dct": dct_features,
        }


In [11]:
test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

In [12]:
test_root = "/kaggle/input/deepfake-ml-challenge/DATASET/test"
test_dataset = CIFakeTestDataset(root=test_root, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False, num_workers=2)

In [13]:
import torch
import torch.nn as nn
from torchvision.models import efficientnet_b1

class SpatialBranch(nn.Module):
    def __init__(self):
        super().__init__()
        # EfficientNet backbone for 256x256 input
        self.backbone = efficientnet_b1(weights="IMAGENET1K_V1")
        self.backbone.classifier = nn.Identity()
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(1280, 512)  # EfficientNet-B1 feature dim → 512 projection
    
    def forward(self, x):
        feats = self.backbone.features(x)    # [B, 1280, 8, 8]
        feats = self.pool(feats).flatten(1)  # [B, 1280]
        return self.fc(feats)                # [B, 512]

class FreqBranch(nn.Module):
    def __init__(self, input_dim=256):
        super().__init__()
        # input: [B, 256]
        # treat it as a 1D signal of length 256 with 1 channel
        self.conv1 = nn.Conv1d(1, 128, kernel_size=3, stride=2, padding=1)
        self.conv2 = nn.Conv1d(128, 64, kernel_size=3, stride=2, padding=1)
        self.conv3 = nn.Conv1d(64, 32, kernel_size=3, stride=2, padding=1)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.fc = nn.Linear(32, 512)
        self.act = nn.GELU()
    
    def forward(self, x):
        x = x.unsqueeze(1)                   # [B, 1, 256]
        x = self.act(self.conv1(x))          # [B, 128, 128]
        x = self.act(self.conv2(x))          # [B, 64, 64]
        x = self.act(self.conv3(x))          # [B, 32, 32]
        x = self.pool(x).flatten(1)          # [B, 32]
        return self.fc(x)                    # [B, 512]

class CrossAttentionFusion(nn.Module):
    def __init__(self, dim=512, heads=4, dropout=0.3):
        super().__init__()
        self.attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim * 2, dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
    
    def forward(self, spatial, freq):
        # [B, 512] → [B, 1, 512]
        spatial = spatial.unsqueeze(1)
        freq = freq.unsqueeze(1)
        attn_out, _ = self.attn(spatial, freq, freq)
        fused = self.norm(attn_out.squeeze(1) + spatial.squeeze(1))
        concat = torch.cat([fused, freq.squeeze(1)], dim=1)  # [B, 1024]
        return self.mlp(concat)  # [B, 512]

class BinaryHead(nn.Module):
    def __init__(self, dim=512, dropout=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(dim, 128), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
    
    def forward(self, x):
        return self.head(x)  # logits

class SpectralClassifier(nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.spatial = SpatialBranch()
        self.freq = FreqBranch(input_dim=256)
        self.fusion = CrossAttentionFusion(dim=512, heads=4, dropout=dropout)
        self.head = BinaryHead(dim=512, dropout=dropout)
    
    def forward(self, img, dct):
        spat = self.spatial(img)     # [B, 512]
        freq = self.freq(dct)        # [B, 512]
        fused = self.fusion(spat, freq)  # [B, 512]
        pred = self.head(fused)      # [B, 1]
        return pred

In [15]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [16]:
model = SpectralClassifier().to(device)
model.load_state_dict(torch.load("/kaggle/input/deepfake-model/pytorch/default/1/deepfake_model_2.pt", map_location=device))
model.eval()

SpectralClassifier(
  (spatial): SpatialBranch(
    (backbone): EfficientNet(
      (features): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): SiLU(inplace=True)
        )
        (1): Sequential(
          (0): MBConv(
            (block): Sequential(
              (0): Conv2dNormActivation(
                (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
                (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (2): SiLU(inplace=True)
              )
              (1): SqueezeExcitation(
                (avgpool): AdaptiveAvgPool2d(output_size=1)
                (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
                (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1,

In [18]:
from tqdm import tqdm

results = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference"):
        images = batch["image"].to(device)
        dcts = batch["dct"].to(device)
        indices = batch["index"]

        outputs = model(images, dcts)
        probs = torch.sigmoid(outputs).squeeze(1)
        preds = (probs > 0.5).int().cpu().tolist()

        for idx, pred in zip(indices, preds):
            results.append({
                "index": int(idx),
                "prediction": "fake" if pred == 1 else "real"
            })

Inference: 100%|██████████| 8/8 [00:02<00:00,  3.14it/s]


In [20]:
import json

with open("inference_predictions.json", "w") as f:
    json.dump(results, f, indent=4)